# 🚗 AI-Powered UK Used Car Valuation System

**Two-stage AI pipeline:**
1. Train a Keras MLP neural network to predict used car prices
2. Use Qwen2.5-3B-Instruct LLM to generate professional explanations of predictions

**Dataset:** 100,000 UK Used Car Dataset by Aditya (Kaggle)

**Workflow:** Data Preprocessing → Model Training → Evaluation → Push to Hugging Face Hub → Inference with LLM Explanation

## 1. Install and Import Dependencies

Install all required libraries and set global random seeds for full reproducibility.

In [ ]:
# ── Install packages (uncomment on Kaggle / Colab if needed) ──
!pip install -q pandas numpy scikit-learn tensorflow matplotlib joblib huggingface_hub transformers torch

In [ ]:
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print(f"Seed       : {SEED}")

## 2. Load and Explore the Dataset

Load the 100,000 UK Used Car Dataset CSV. Inspect shape, dtypes, head, describe, and missing values.

> **Kaggle path hint:** On Kaggle the CSV files are typically under `/kaggle/input/100000-uk-used-car-data-set/`. Update the glob below if your input folder differs.

In [ ]:
# ── Load all CSVs in the Kaggle input folder and concatenate ──
import glob

# Auto-detect: search all subdirectories under /kaggle/input/ for CSVs
DATA_DIR = "/kaggle/input/"
csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found under {DATA_DIR}. "
        "Make sure you added the '100,000 UK Used Car Data Set' to this notebook."
    )

print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"  • {f}")

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"\nCombined shape: {df.shape}")
df.head()

In [ ]:
# ── Basic exploration ──
print("Columns:", list(df.columns))
print("\nDtypes:")
print(df.dtypes)
print("\nDescribe:")
df.describe()

In [ ]:
# ── Missing values ──
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")

## 3. Data Cleaning and Feature Selection

Drop irrelevant columns (`tax`, `tax(£)`, `mpg`, or any unnamed index columns). Remove rows where `price` is missing or zero. Standardise column naming so the downstream pipeline works consistently.

In [ ]:
# ── Column name cleanup ──
# Some CSVs have leading/trailing spaces or varying cases
df.columns = df.columns.str.strip()

# The dataset may have a manufacturer column instead of 'brand'
# Normalise: if a column looks like the brand, rename it
if "brand" not in df.columns:
    # Common alternatives in this dataset
    for candidate in ["manufacturer", "make"]:
        if candidate in df.columns:
            df.rename(columns={candidate: "brand"}, inplace=True)
            print(f"Renamed '{candidate}' → 'brand'")
            break

# ── Drop irrelevant columns ──
DROP_COLS = ["tax", "tax(£)", "mpg"]
existing_drop = [c for c in DROP_COLS if c in df.columns]
# Also drop any unnamed index columns
unnamed = [c for c in df.columns if "unnamed" in c.lower()]
to_drop = existing_drop + unnamed
if to_drop:
    df.drop(columns=to_drop, inplace=True)
    print(f"Dropped columns: {to_drop}")

# ── Coerce numeric columns (price, year, mileage, engineSize may be strings) ──
for col in ["price", "year", "mileage", "engineSize"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print(f"Coerced numeric columns to proper dtype")

# ── Remove rows with missing or zero price ──
before = len(df)
df.dropna(subset=["price"], inplace=True)
df = df[df["price"] > 0].copy()
print(f"Removed {before - len(df)} rows with missing/zero price")

# ── Keep only the core columns ──
CORE_COLS = ["price", "year", "mileage", "engineSize", "fuelType", "transmission", "model", "brand"]
available = [c for c in CORE_COLS if c in df.columns]
df = df[available].copy()

print(f"\nCleaned dataset shape: {df.shape}")
print(f"Columns kept: {list(df.columns)}")
df.head()

## 4. Target Variable Transformation (log1p)

Apply `numpy.log1p` to the price column to reduce right-skewness and improve neural network convergence. Visualise before/after distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before transformation
axes[0].hist(df["price"], bins=100, color="steelblue", edgecolor="white")
axes[0].set_title("Price Distribution (Original)")
axes[0].set_xlabel("Price (£)")
axes[0].set_ylabel("Count")

# Apply log1p
df["price"] = np.log1p(df["price"])

# After transformation
axes[1].hist(df["price"], bins=100, color="coral", edgecolor="white")
axes[1].set_title("Price Distribution (log1p Transformed)")
axes[1].set_xlabel("log1p(Price)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Target column 'price' is now log1p-transformed.")

## 5. Temporal Train/Test Split

Split data temporally: **train on year ≤ 2018**, **test on year > 2018**. This prevents data leakage from future observations into the training set and better simulates real-world deployment.

In [ ]:
SPLIT_YEAR = 2018

train_df = df[df["year"] <= SPLIT_YEAR].copy()
test_df  = df[df["year"] >  SPLIT_YEAR].copy()

TARGET = "price"

y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values

X_train_df = train_df.drop(columns=[TARGET])
X_test_df  = test_df.drop(columns=[TARGET])

print(f"Train set : {X_train_df.shape[0]:,} rows  (year ≤ {SPLIT_YEAR})")
print(f"Test  set : {X_test_df.shape[0]:,} rows  (year >  {SPLIT_YEAR})")
print(f"Train target range: [{y_train.min():.2f}, {y_train.max():.2f}]")
print(f"Test  target range: [{y_test.min():.2f},  {y_test.max():.2f}]")

## 6. One-Hot Encoding and Column Alignment

One-hot encode the categorical columns (`fuelType`, `transmission`, `model`, `brand`) and align test columns to match train.

In [ ]:
CATEGORICAL_COLS = ["fuelType", "transmission", "model", "brand"]
cols_to_encode = [c for c in CATEGORICAL_COLS if c in X_train_df.columns]

X_train_df = pd.get_dummies(X_train_df, columns=cols_to_encode, dtype=float)
X_test_df  = pd.get_dummies(X_test_df,  columns=cols_to_encode, dtype=float)

# Align: keep only columns present in training, fill missing test cols with 0
X_train_df, X_test_df = X_train_df.align(X_test_df, join="left", axis=1, fill_value=0)

# Store the feature column order (needed for inference later)
FEATURE_COLUMNS = list(X_train_df.columns)

print(f"Features after one-hot encoding: {len(FEATURE_COLUMNS)}")
print(f"X_train shape: {X_train_df.shape}")
print(f"X_test  shape: {X_test_df.shape}")

## 7. Feature Scaling with StandardScaler

Fit a `StandardScaler` on the training features and transform both splits. Save the scaler for deployment.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df.values)
X_test  = scaler.transform(X_test_df.values)

# Save scaler
joblib.dump(scaler, "scaler.joblib")
print("Saved scaler → scaler.joblib")

# Quick sanity check
print(f"\nScaled training features — mean: {X_train.mean(axis=0)[:5].round(6)}")
print(f"Scaled training features — std : {X_train.std(axis=0)[:5].round(4)}")

## 8. Build Keras MLP Neural Network

Architecture:
- Dense(64, ReLU)
- Dense(32, ReLU)
- Dense(1, Linear)

Compiled with Adam optimiser, MSE loss, and MAE metric.

In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1,  activation="linear"),
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

model.summary()

## 9. Train the Model with EarlyStopping

Train for up to 50 epochs with `batch_size=32` and `validation_split=0.2`. EarlyStopping monitors `val_loss` with `patience=5` and restores the best weights.

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1,
)

print(f"\nTraining stopped at epoch {len(history.history['loss'])}")

## 10. Evaluate Model Performance (MAE, RMSE, R²)

Generate predictions on the test set, convert back to real price scale with `expm1`, and compute regression metrics.

In [ ]:
# ── Predict on test set ──
y_pred_log = model.predict(X_test, verbose=0).flatten()

# ── Convert back to real price scale ──
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# ── Metrics ──
mae  = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2   = r2_score(y_test_real, y_pred_real)

print("=" * 50)
print("  Test Set Evaluation (Real £ Scale)")
print("=" * 50)
print(f"  MAE  : £{mae:,.2f}")
print(f"  RMSE : £{rmse:,.2f}")
print(f"  R²   : {r2:.4f}")
print("=" * 50)
print(f"\nInterpretation: The model's average prediction error is ~£{mae:,.0f}.")
print(f"R² of {r2:.4f} means the model explains {r2*100:.1f}% of price variance.")

## 11. Visualise Training Loss Curves

Plot training and validation loss over epochs to diagnose overfitting and confirm EarlyStopping behaviour.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history["loss"],     label="Train Loss", linewidth=2)
axes[0].plot(history.history["val_loss"],  label="Val Loss",   linewidth=2)
axes[0].set_title("Loss (MSE) over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history["mae"],      label="Train MAE", linewidth=2)
axes[1].plot(history.history["val_mae"],   label="Val MAE",   linewidth=2)
axes[1].set_title("MAE over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE (log scale)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Predicted vs Actual Scatter Plot

Scatter plot of predicted vs actual prices (real £ scale). The diagonal line represents perfect predictions.

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test_real, y_pred_real, alpha=0.15, s=10, color="steelblue")

# Diagonal reference line
max_val = max(y_test_real.max(), y_pred_real.max())
plt.plot([0, max_val], [0, max_val], "r--", linewidth=2, label="Perfect Prediction")

plt.title(f"Predicted vs Actual Price  (R² = {r2:.4f})")
plt.xlabel("Actual Price (£)")
plt.ylabel("Predicted Price (£)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Save Model and Artefacts

Save the trained Keras model, confirm the scaler, and create `model_config.json` with the feature column names.

In [ ]:
# ── Save Keras model ──
MODEL_PATH = "car_price_model.keras"
model.save(MODEL_PATH)
print(f"Saved model → {MODEL_PATH}")

# ── Save model config (feature columns) ──
CONFIG_PATH = "model_config.json"
config = {"feature_columns": FEATURE_COLUMNS}
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved config → {CONFIG_PATH}  ({len(FEATURE_COLUMNS)} features)")

# ── Verify all artefacts ──
artefacts = [MODEL_PATH, "scaler.joblib", CONFIG_PATH]
print("\nSaved artefacts:")
for a in artefacts:
    size = os.path.getsize(a) / 1024
    print(f"  ✓ {a}  ({size:.1f} KB)")

## 14. Push Artefacts to Hugging Face Hub

Login with your HF token, create the repo, and upload all three artefacts.

> **Before running:** Set your Hugging Face token as `HF_TOKEN` environment variable or Kaggle secret.

In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

# ── Authenticate ──
# Try Kaggle secrets first, then env variable
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)
    if HF_TOKEN:
        print("Loaded HF token from environment variable.")
    else:
        print("⚠ No HF_TOKEN found. Set it as a Kaggle secret or env variable.")

if HF_TOKEN:
    login(token=HF_TOKEN)

    # ── Set your repo name here ──
    REPO_ID = "raghavverma271/uk-used-car-nn"

    api = HfApi()

    # Create repo if it doesn't exist
    api.create_repo(repo_id=REPO_ID, exist_ok=True, repo_type="model")
    print(f"Repository ready: https://huggingface.co/{REPO_ID}")

    # Upload artefacts
    for filepath in ["car_price_model.keras", "scaler.joblib", "model_config.json"]:
        api.upload_file(
            path_or_fileobj=filepath,
            path_in_repo=filepath,
            repo_id=REPO_ID,
            repo_type="model",
        )
        print(f"  Uploaded: {filepath}")

    print(f"\nAll artefacts pushed to https://huggingface.co/{REPO_ID}")
else:
    print("Skipping upload — no token available.")

## 15. Build Local Adapter Loader (`adapter.py`)

Write the adapter module that downloads artefacts from HF Hub and loads them for inference.

In [ ]:
%%writefile adapter.py
"""
adapter.py — Hugging Face Hub adapter for the Used Car Valuation model.

Downloads the trained Keras model, scaler, and feature configuration
from a Hugging Face repository and returns ready-to-use objects.
"""

import json
import os
from typing import Tuple, Dict, List, Any

import joblib
import numpy as np
from huggingface_hub import hf_hub_download
from tensorflow import keras

# Default artefact filenames (must match what the training notebook uploads)
MODEL_FILENAME = "car_price_model.keras"
SCALER_FILENAME = "scaler.joblib"
CONFIG_FILENAME = "model_config.json"


def load_model_from_hf(
    repo_id: str,
    cache_dir: str = None,
    token: str = None,
) -> Tuple[keras.Model, Any, List[str]]:
    """
    Download model artefacts from Hugging Face Hub and load them.

    Parameters
    ----------
    repo_id : str
        Hugging Face repository identifier, e.g. "username/uk-used-car-nn".
    cache_dir : str, optional
        Local directory to cache downloaded files.
    token : str, optional
        Hugging Face access token.

    Returns
    -------
    model : keras.Model
    scaler : StandardScaler
    feature_columns : list[str]
    """
    model_path = hf_hub_download(
        repo_id=repo_id, filename=MODEL_FILENAME,
        cache_dir=cache_dir, token=token,
    )
    scaler_path = hf_hub_download(
        repo_id=repo_id, filename=SCALER_FILENAME,
        cache_dir=cache_dir, token=token,
    )
    config_path = hf_hub_download(
        repo_id=repo_id, filename=CONFIG_FILENAME,
        cache_dir=cache_dir, token=token,
    )

    model = keras.models.load_model(model_path)
    scaler = joblib.load(scaler_path)

    with open(config_path, "r") as f:
        config = json.load(f)
    feature_columns = config["feature_columns"]

    print(f"[adapter] Loaded model, scaler, and config from '{repo_id}'")
    print(f"[adapter] Feature count: {len(feature_columns)}")
    return model, scaler, feature_columns

## 16. Build Lightweight Local Frontend (`inference.py`)

Write the local inference module. This is a **bare frontend** — no local LLM loading.
- Price prediction: runs the tiny NN locally on CPU (instant).
- LLM explanation: calls Qwen via **HF Inference API** (serverless, no local GPU needed).

In [ ]:
%%writefile inference.py
"""
inference.py — Lightweight local frontend for the Used Car Valuation System.

All heavy computation lives on Kaggle / HF Hub. Locally we only:
  1. Download the tiny NN + scaler from HF (cached after first use)
  2. Run a single forward-pass for price prediction (CPU, instant)
  3. Call HF Inference API for Qwen LLM explanation (serverless)

Usage:
    python inference.py \
        --repo  username/uk-used-car-nn \
        --brand BMW --model "3 Series" --year 2019 \
        --mileage 30000 --engineSize 2.0 \
        --fuelType Diesel --transmission Automatic
"""

import argparse
import os
from typing import Any, Dict

import numpy as np
import pandas as pd
from adapter import load_model_from_hf

CATEGORICAL_COLS = ["fuelType", "transmission", "model", "brand"]

PROMPT_TEMPLATE = """Car Details:
Brand: {brand}
Model: {model}
Year: {year}
Mileage: {mileage}
Engine Size: {engine}
Fuel Type: {fuel}
Transmission: {transmission}

Predicted Market Price: £{price}

Explain clearly and professionally why this vehicle is valued this way.
Discuss depreciation, mileage, engine size, and brand positioning."""


def preprocess_single(car_dict, feature_columns, scaler):
    df = pd.DataFrame([car_dict])
    cols_to_encode = [c for c in CATEGORICAL_COLS if c in df.columns]
    df = pd.get_dummies(df, columns=cols_to_encode, dtype=float)
    df = df.reindex(columns=feature_columns, fill_value=0)
    return scaler.transform(df.values)


def generate_explanation(car_dict, predicted_price, hf_token=None):
    """Call Qwen via HF Inference API — no local GPU needed."""
    from huggingface_hub import InferenceClient

    token = hf_token or os.environ.get("HF_TOKEN")
    if not token:
        return "(Set HF_TOKEN to enable LLM explanations)"

    client = InferenceClient(token=token)
    prompt = PROMPT_TEMPLATE.format(
        brand=car_dict.get("brand", "Unknown"),
        model=car_dict.get("model", "Unknown"),
        year=car_dict.get("year", "N/A"),
        mileage=car_dict.get("mileage", "N/A"),
        engine=car_dict.get("engineSize", "N/A"),
        fuel=car_dict.get("fuelType", "N/A"),
        transmission=car_dict.get("transmission", "N/A"),
        price=f"{predicted_price:,.2f}",
    )
    messages = [
        {"role": "system", "content": "You are an expert automotive valuation analyst."},
        {"role": "user", "content": prompt},
    ]
    response = client.chat_completion(
        model="Qwen/Qwen2.5-3B-Instruct", messages=messages, max_tokens=150,
    )
    return response.choices[0].message.content.strip()


def predict_and_explain(car_dict, model, scaler, feature_columns,
                        explain=True, hf_token=None):
    X = preprocess_single(car_dict, feature_columns, scaler)
    log_price = model.predict(X, verbose=0)[0][0]
    predicted_price = float(np.expm1(log_price))
    result = {"predicted_price": round(predicted_price, 2)}
    if explain:
        result["explanation_text"] = generate_explanation(
            car_dict, predicted_price, hf_token=hf_token)
    return result


def main():
    parser = argparse.ArgumentParser(description="Predict used-car price.")
    parser.add_argument("--repo", type=str, required=True, help="HF repo id")
    parser.add_argument("--brand", type=str, required=True)
    parser.add_argument("--model", type=str, required=True, dest="car_model")
    parser.add_argument("--year", type=int, required=True)
    parser.add_argument("--mileage", type=int, required=True)
    parser.add_argument("--engineSize", type=float, required=True)
    parser.add_argument("--fuelType", type=str, required=True)
    parser.add_argument("--transmission", type=str, required=True)
    parser.add_argument("--no-explain", action="store_true", default=False)
    parser.add_argument("--token", type=str, default=None)
    args = parser.parse_args()

    car_dict = {
        "brand": args.brand, "model": args.car_model,
        "year": args.year, "mileage": args.mileage,
        "engineSize": args.engineSize, "fuelType": args.fuelType,
        "transmission": args.transmission,
    }

    print("=" * 60)
    print("  UK Used Car Valuation System")
    print("=" * 60)
    nn_model, scaler, feature_columns = load_model_from_hf(args.repo, token=args.token)
    token = args.token or os.environ.get("HF_TOKEN")
    result = predict_and_explain(
        car_dict, nn_model, scaler, feature_columns,
        explain=not args.no_explain, hf_token=token,
    )
    print(f"\n  Predicted Price : £{result['predicted_price']:,.2f}")
    if "explanation_text" in result:
        print(f"\n  Explanation:\n  {result['explanation_text']}")
    print("=" * 60)


if __name__ == "__main__":
    main()

## 17. Load Qwen LLM for Price Explanations

Load **Qwen/Qwen2.5-3B-Instruct** with `float16` precision. This cell requires a GPU runtime on Kaggle (enable GPU in Settings → Accelerator).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline

LLM_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {LLM_MODEL_ID} ...")
print(f"CUDA available: {torch.cuda.is_available()}")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

llm_pipe = hf_pipeline("text-generation", model=llm_model, tokenizer=tokenizer)
print("Qwen LLM loaded successfully.")

# ── Quick test ──
test_out = llm_pipe(
    [{"role": "user", "content": "Say hello in one sentence."}],
    max_new_tokens=30, do_sample=False,
)
print(f"Test output: {test_out[0]['generated_text'][-1]['content']}")

## 18. End-to-End Prediction with Explanation

Run a complete test: define a sample car, predict price with the trained NN, and generate a professional explanation with Qwen.

In [ ]:
# ── Sample car for testing ──
sample_car = {
    "brand": "BMW",
    "model": "3 Series",
    "year": 2019,
    "mileage": 30000,
    "engineSize": 2.0,
    "fuelType": "Petrol",
    "transmission": "Automatic",
}

# ── Preprocess ──
sample_df = pd.DataFrame([sample_car])
cols_to_encode = [c for c in CATEGORICAL_COLS if c in sample_df.columns]
sample_df = pd.get_dummies(sample_df, columns=cols_to_encode, dtype=float)
sample_df = sample_df.reindex(columns=FEATURE_COLUMNS, fill_value=0)
X_sample = scaler.transform(sample_df.values)

# ── Predict ──
log_pred = model.predict(X_sample, verbose=0)[0][0]
pred_price = float(np.expm1(log_pred))

print(f"Sample Car: {sample_car['year']} {sample_car['brand']} {sample_car['model']}")
print(f"Predicted Price: £{pred_price:,.2f}\n")

# ── Generate explanation with Qwen ──
PROMPT_TEMPLATE = """Car Details:
Brand: {brand}
Model: {model}
Year: {year}
Mileage: {mileage}
Engine Size: {engine}
Fuel Type: {fuel}
Transmission: {transmission}

Predicted Market Price: £{price}

Explain clearly and professionally why this vehicle is valued this way.
Discuss depreciation, mileage, engine size, and brand positioning."""

prompt = PROMPT_TEMPLATE.format(
    brand=sample_car["brand"], model=sample_car["model"],
    year=sample_car["year"], mileage=sample_car["mileage"],
    engine=sample_car["engineSize"], fuel=sample_car["fuelType"],
    transmission=sample_car["transmission"],
    price=f"{pred_price:,.2f}",
)

messages = [
    {"role": "system", "content": "You are an expert automotive valuation analyst."},
    {"role": "user", "content": prompt},
]

result = llm_pipe(messages, max_new_tokens=150, do_sample=False)
explanation = result[0]["generated_text"][-1]["content"].strip()

print("=" * 60)
print("  VALUATION REPORT")
print("=" * 60)
print(f"  Vehicle : {sample_car['year']} {sample_car['brand']} {sample_car['model']}")
print(f"  Price   : £{pred_price:,.2f}")
print(f"\n  Explanation:\n")
print(f"  {explanation}")
print("=" * 60)

## 19. Utility Module (`utils.py`)

Write the shared utility module used by the project files for consistent preprocessing.

In [ ]:
%%writefile utils.py
"""
utils.py — Shared utility functions for the Used Car Valuation System.
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from typing import Tuple, List, Dict, Any

CATEGORICAL_COLS = ["fuelType", "transmission", "model", "brand"]
NUMERIC_COLS = ["year", "mileage", "engineSize"]
TARGET_COL = "price"
TEMPORAL_SPLIT_YEAR = 2018
DROP_COLS = ["tax", "tax(£)", "mpg"]
RANDOM_SEED = 42


def load_and_clean(filepath: str) -> pd.DataFrame:
    """Load CSV, drop irrelevant columns, and remove rows with missing price."""
    df = pd.read_csv(filepath)
    col_map = {c: c.strip() for c in df.columns}
    df.rename(columns=col_map, inplace=True)
    existing_drop = [c for c in DROP_COLS if c in df.columns]
    if existing_drop:
        df.drop(columns=existing_drop, inplace=True)
    df.dropna(subset=[TARGET_COL], inplace=True)
    df = df[df[TARGET_COL] > 0].copy()
    return df


def apply_log_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TARGET_COL] = np.log1p(df[TARGET_COL])
    return df


def temporal_split(df, split_year=TEMPORAL_SPLIT_YEAR):
    train = df[df["year"] <= split_year].copy()
    test = df[df["year"] > split_year].copy()
    return train, test


def one_hot_encode(train, test, categorical_cols=None):
    if categorical_cols is None:
        categorical_cols = CATEGORICAL_COLS
    cols_to_encode = [c for c in categorical_cols if c in train.columns]
    train = pd.get_dummies(train, columns=cols_to_encode, dtype=float)
    test = pd.get_dummies(test, columns=cols_to_encode, dtype=float)
    train, test = train.align(test, join="left", axis=1, fill_value=0)
    return train, test


def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


def preprocess_single(car_dict, feature_columns, scaler):
    df = pd.DataFrame([car_dict])
    cols_to_encode = [c for c in CATEGORICAL_COLS if c in df.columns]
    df = pd.get_dummies(df, columns=cols_to_encode, dtype=float)
    df = df.reindex(columns=feature_columns, fill_value=0)
    return scaler.transform(df.values)


def compute_metrics(y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

## 20. Summary

### Architecture Split

| Component | Where it runs | Why |
|---|---|---|
| NN Training (Keras MLP) | **Kaggle** (this notebook) | GPU/CPU heavy, one-time |
| Qwen LLM demo | **Kaggle** (this notebook) | GPU required for local loading |
| Push model to HF Hub | **Kaggle** (this notebook) | Direct upload after training |
| Price prediction | **Local** (CPU) | Tiny NN, instant forward pass |
| LLM explanation | **HF Inference API** | Serverless, zero local GPU |
| Frontend (CLI / Streamlit) | **Local** | Bare UI, takes input → shows output |

### What this notebook produced:
- Trained Keras MLP on 100K UK used cars
- Evaluated: MAE, RMSE, R² on real-price scale
- Pushed `car_price_model.keras`, `scaler.joblib`, `model_config.json` to HF Hub
- Generated `adapter.py`, `inference.py`, `utils.py` for local use
- Demonstrated Qwen2.5-3B-Instruct end-to-end explanation

### To run locally (no GPU needed):

**CLI:**
```bash
python inference.py \
    --repo YOUR_USERNAME/uk-used-car-nn \
    --brand BMW --model "3 Series" --year 2019 \
    --mileage 30000 --engineSize 2.0 \
    --fuelType Petrol --transmission Automatic \
    --token hf_your_token
```

**Streamlit UI:**
```bash
pip install streamlit
streamlit run app.py
```